# Repository Momentum Dashboard

This notebook presents a repository-level momentum analysis using public GitHub sample commit data. The goal is to identify repositories showing stronger short-term activity signals by combining recent commit activity, growth rate, and contributor participation.

The dashboard is exploratory and portfolio-focused. It does not claim to measure real-time GitHub trends, topic-level technology momentum, stars, forks, issues, pull requests, or production-grade repository health. Instead, it demonstrates SQL-based feature engineering, normalized scoring, and Python/Plotly dashboard storytelling.

## Setup and Data Loading

In [1]:
import os
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "iframe"

normalized_df = pd.read_csv("../data/processed/normalized_momentum_score.csv")
radar_df = pd.read_csv("../data/processed/repository_discovery_radar.csv")

os.makedirs("../docs/assets/charts", exist_ok=True)

In [2]:
normalized_df["growth_rate_label"] = normalized_df["growth_rate"].apply(
    lambda x: f"{x:+.2f}%" if pd.notnull(x) else "New/Emerging"
)

normalized_df["growth_rate_plot"] = normalized_df["growth_rate"].fillna(0)

min_score = normalized_df["momentum_score"].min()
normalized_df["visual_size"] = normalized_df["momentum_score"] - min_score + 1

# Repository Momentum Dashboard

## Chart 1: Repository Momentum Ranking

This chart ranks repositories by normalized momentum score. The score combines recent commit activity, growth rate, and contributor participation to create a balanced view of repository-level momentum.

Rather than relying only on raw commit volume, this view helps compare repositories with different activity levels and highlights which repositories show stronger short-term activity signals in the sample dataset.

In [3]:
chart_df= normalized_df.sort_values("momentum_score", ascending=True)


fig=px.bar(chart_df,x="momentum_score", 
           y="repo_name", title="Repository Momentum Ranking",orientation='h', 
           color="activity_status",
          color_discrete_map={
  "Growing": "#6366F1",
  "Stable" :  "#CBD5E1",
  "New/Emerging": "#22D3EE"
},
custom_data=["repo_name","momentum_score", 
             "growth_rate_label", "contributor_count", "activity_status", "commits_last_30_days"]
)


fig.update_layout(template="plotly_white", xaxis_title="Normalized Momentum Score", 
                  yaxis_title="Repository",
                  legend_title="Activity Status",
                 yaxis=dict(
    categoryorder="array",
    categoryarray=chart_df["repo_name"].tolist()
),
legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.35,
        xanchor="center",
        x=0.5,
maxheight=0.1),
                 margin=dict(b=100),
                  height=650,
width=1200
                 
)


fig.update_traces( texttemplate='%{x:.2f}', # Shows the x-value as the label 
                   textposition='outside', # Places the label next to the bar
                   hovertemplate="<b>%{customdata[0]}</b><br><br>" + 
                   "Momentum Score: %{customdata[1]:.2f}<br>" + 
                   "Growth Rate: %{customdata[2]}<br>" + "Contributors: %{customdata[3]}<br><br>" 
                   + "Status: %{customdata[4]}" + "<extra></extra>" )

fig


In [4]:
fig.write_image("../docs/assets/charts/01_repository_momentum_ranking.png", width=1400, height=800)

### Key Insight

TensorFlow and VS Code show the strongest momentum signals, while Linux demonstrates that a large contributor base does not necessarily translate into positive short-term momentum.

## Chart 2: What Drives Repository Momentum?

While Chart 1 identifies the highest-momentum repositories, this visualization explains the drivers behind that momentum. By comparing repository growth, contributor participation, and recent commit activity, we can distinguish between repositories with broad contributor bases, repositories with strong short-term growth, and repositories whose activity is more concentrated in one signal.

This helps make the momentum score more interpretable rather than treating it as a black-box ranking.

In [5]:
fig2=px.scatter(normalized_df, x="growth_rate_plot", y="contributor_count", 
                title="What Drives Repository Momentum?", 
           color="activity_status", labels={'growth_rate_plot':'Growth Rate (%)', 
                                            'contributor_count':'Contributor Count', 
                                            'activity_status':'Activity Status'},
                size="visual_size",  size_max=40,  text="repo_name",
          color_discrete_map={
  "Growing": "#6366F1",
  "Stable" :  "#CBD5E1",
  "New/Emerging": "#22D3EE"
},
          custom_data=["repo_name","momentum_score",
                       "growth_rate_label",
                       "contributor_count", "activity_status", "commits_last_30_days"]
)



fig2.update_layout(template="plotly_white", xaxis_title= "Growth Rate (%)", 
                  yaxis_title="Contributor Count",
                  legend_title="Activity Status",
legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.35,
        xanchor="center",
        x=0.5,
maxheight=0.1),
                 margin=dict(b=100),
                  height=650,
width=1200
                 
)


fig2.update_traces( textposition="top center",  
                    hovertemplate="<b>%{customdata[0]}</b><br><br>"  +
    "Growth Rate: %{customdata[2]}<br>" + "Contributors: %{customdata[3]}<br><br>"
                    + "Recent Commits: %{customdata[5]}<br>"
                    + "Momentum Score: %{customdata[1]:.2f}<br>"
    + "Status: %{customdata[4]}" + "<extra></extra>", cliponaxis=False )


fig2.add_vline(x=0, 
    line_width=2, 
    line_dash="dash", 
    line_color="#EF4444",  # Soft Red 
               annotation_text="Growth = 0%",
    annotation_position="top left"
)



fig2


In [6]:
fig2.write_image("../docs/assets/charts/02_repository_momentum_drivers.png", width=1400, height=800)

### Key Insight

Repository momentum is shaped by more than one signal. TensorFlow combines strong short-term growth with meaningful contributor participation, while VS Code also shows positive growth but with a smaller contributor base in this sample window.

Linux shows the opposite pattern: it has the largest contributor count, but negative short-term growth, which explains why a large community alone does not automatically translate into high recent momentum.

Bootstrap appears as a New/Emerging case in this sample window because it has recent activity but limited prior-period baseline.

## Chart 3: Momentum Driver Fingerprint

This heatmap breaks each repository’s momentum score into its underlying components: growth, recent commit activity, and contributor participation.

Each row represents a repository, and each column represents one score component. Cell color shows both the direction and strength of each component’s contribution, making it easier to see whether a repository’s momentum is driven by growth, recent activity volume, or contributor breadth.

In [7]:

map_df= (
    normalized_df[["repo_name", "momentum_score", "growth_component", "recent_activity_component", 
                       "contributor_component"]]
                            .sort_values("momentum_score", ascending=False)
                                .set_index("repo_name")
        )

component_map_df= (
    map_df[["growth_component", "recent_activity_component", "contributor_component"]]
    .rename(columns={"growth_component": "Growth", "recent_activity_component": "Recent Activity", 
                     "contributor_component":"Contributor"})
)


fig3=px.imshow(
    component_map_df,
               text_auto=".2f", title="<b>Momentum Driver Fingerprint</b>",
               labels=dict(x="Component", y="Repository", color="<b>Contribution</b>"),
          color_continuous_scale="RdBu",
               color_continuous_midpoint=0,
         aspect='auto'

)


fig3.update_layout(
    template = "plotly_white", 
    yaxis_title = "<b>Repository</b>", 
                xaxis_title = "<b>Score Component</b>",
                  height=600,
                   width=1050,
    margin=dict(r=130, b=120),
    coloraxis_colorbar=dict(
           orientation="h", # Makes the colorbar horizontal
        y=-0.4,
        thickness=18,
        tickformat=".2f"
    )
)



fig3.update_xaxes(
    title_font=dict(color="#1f2937", size=13),
    tickmode="array",
    tickvals=component_map_df.columns.tolist(),
    ticktext=[f"<b>{col}</b>" for col in component_map_df.columns],
    tickfont=dict(
        color="#1f2937",
        size=11
    )
)




fig3.update_yaxes(
    title_font=dict(color="#1f2937", size=13),
    tickfont=dict(
        color="#334155",
        size=11
    )
)




fig3.update_traces(
    hovertemplate="Repository: %{y}<br>" 
    + "Component: %{x}<br><b>"
    + "contribution value: %{z:.2f}</b><extra></extra>"
)


fig3.show(config={"displayModeBar": False})

In [8]:
fig3.write_image("../docs/assets/charts/03_momentum_driver_fingerprint.png", width=1400, height=800)

### Key Insight

The heatmap shows that repository momentum is built from different combinations of growth, recent activity, and contributor participation. TensorFlow’s high momentum score is primarily driven by its strong growth component, while VS Code shows a more balanced mix of growth and recent activity.

Linux has strong recent activity and contributor participation, but its negative growth component pulls down its overall momentum score. This explains why a large contributor base alone does not automatically translate into high short-term momentum.

React also shows a negative growth contribution in this sample window, while Bootstrap has only minimal component contribution across the three score drivers. This makes the component view useful for interpreting not just the top-ranked repositories, but also why lower-ranked repositories receive weaker momentum scores.